# Data Importing

In [23]:
!pip install --upgrade pandas numpy


In [24]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = 'data/'
orders      = pd.read_csv(DATA + 'orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv(DATA + 'order_items.csv')
products    = pd.read_csv(DATA + 'products.csv')
customers   = pd.read_csv(DATA + 'customers.csv')
geography   = pd.read_csv(DATA + 'geography.csv')
returns_df  = pd.read_csv(DATA + 'returns.csv')
payments    = pd.read_csv(DATA + 'payments.csv')
web_traffic = pd.read_csv(DATA + 'web_traffic.csv', parse_dates=['date'])
sales       = pd.read_csv(DATA + 'sales.csv',       parse_dates=['Date'])

print(f'  orders:      {len(orders):,} rows')
print(f'  order_items: {len(order_items):,} rows')
print(f'  products:    {len(products):,} rows')
print(f'  customers:   {len(customers):,} rows')
print(f'  returns:     {len(returns_df):,} rows')
print(f'  payments:    {len(payments):,} rows')

  orders:      646,945 rows
  order_items: 714,669 rows
  products:    2,412 rows
  customers:   121,930 rows
  returns:     39,939 rows
  payments:    646,945 rows


Câu 1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

A) 30 ngày      B) 90 ngày      C) 180 ngày      D) 365 ngày


In [25]:
# Lọc bỏ các đơn hàng bị Hủy (Cancelled) hoặc Trả lại (Returned)
valid_orders = orders[~orders["order_status"].str.lower().isin(["cancelled", "returned"])].copy()

# Lấy phần Date (dùng dt.normalize() để giữ nguyên kiểu datetime của Pandas)
valid_orders["date_only"] = valid_orders["order_date"].dt.normalize()

# Bỏ trùng lặp: nếu 1 khách mua 2 đơn cùng 1 ngày, ta chỉ tính là 1 lần mua
unique_purchases = valid_orders.drop_duplicates(subset=["customer_id", "date_only"]).sort_values(["customer_id", "date_only"])

# Tính ngày mua liền trước 
unique_purchases["prev_date"] = unique_purchases.groupby("customer_id")["date_only"].shift(1)

# Tính khoảng cách (gap) và ép về số ngày (.dt.days)
unique_purchases["gap_days"] = (unique_purchases["date_only"] - unique_purchases["prev_date"]).dt.days
gaps = unique_purchases["gap_days"].dropna()

print(f"Số cặp đơn liên tiếp (hợp lệ): {len(gaps):,}")
print(f"Trung vị gap (median): {gaps.median():.1f} ngày")
print(f"Trung bình gap (mean): {gaps.mean():.1f} ngày")
print(f"25th Percentile: {gaps.quantile(0.25):.1f} ngày")
print(f"75th Percentile: {gaps.quantile(0.75):.1f} ngày")


Số cặp đơn liên tiếp (hợp lệ): 461,003
Trung vị gap (median): 168.0 ngày
Trung bình gap (mean): 316.8 ngày
25th Percentile: 56.0 ngày
75th Percentile: 399.0 ngày


In [26]:
print('Chọn đáp án C')

Chọn đáp án C


Câu 2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price−cogs)/price?

A) Premium      B) Performance      C) Activewear      D) Standard


In [27]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

margin_by_segment = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print('Tỷ suất lợi nhuận gộp trung bình theo segment:')
print(margin_by_segment.round(4).to_string())
print()
print(f'Cao nhất: {margin_by_segment.idxmax()} ({margin_by_segment.max():.4f})')

Tỷ suất lợi nhuận gộp trung bình theo segment:
segment
Standard       0.3134
Premium        0.2854
All-weather    0.2842
Activewear     0.2656
Performance    0.2636
Balanced       0.2580
Trendy         0.2408
Everyday       0.2363

Cao nhất: Standard (0.3134)


In [28]:
print('Chọn đáp án D')

Chọn đáp án D


Câu 3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A) defective      B) wrong_size      C) changed_mind      D) not_as_described


In [29]:
returns_products = returns_df.merge(products[['product_id', 'category']], on='product_id', how='left')
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']
reason_counts = streetwear_returns['return_reason'].value_counts()

print(f'Số bản ghi trả hàng Streetwear: {len(streetwear_returns):,}')
print()
print('Lý do trả hàng trong Streetwear:')
print(reason_counts.to_string())


Số bản ghi trả hàng Streetwear: 21,799

Lý do trả hàng trong Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159


In [30]:
print('Chọn đáp án B')

Chọn đáp án B


Câu 4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất?

A) organic_search      B) paid_search      C) email_campaign      D) social_media


In [31]:
bounce_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print('Bounce rate trung bình theo traffic_source:')
print(bounce_by_source.to_string())
print()
print(f'Nguồn Traffic có tỉ lệ bounce thấp nhất: {bounce_by_source.idxmin()} ({bounce_by_source.min():.6f})')

Bounce rate trung bình theo traffic_source:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511

Nguồn Traffic có tỉ lệ bounce thấp nhất: email_campaign (0.004458)


In [32]:
print('Chọn đáp án A')

Chọn đáp án A


Câu 5: Tỷ lệ các dòng trong order_items.csv có áp dụng khuyến mãi (promo_id không null) xấp xỉ là bao nhiêu?

A) 12%      B) 25%      C) 39%      D) 54%


In [33]:
total_rows = len(order_items)
promo_rows = order_items['promo_id'].notna().sum()
promo_rate = promo_rows / total_rows * 100

print(f'Tổng số dòng order_items : {total_rows:,}')
print(f'Dòng có promo_id (≠ null): {promo_rows:,}')
print(f'Tỷ lệ có khuyến mãi      : {promo_rate:.2f}%')

Tổng số dòng order_items : 714,669
Dòng có promo_id (≠ null): 276,316
Tỷ lệ có khuyến mãi      : 38.66%


In [34]:
print('Chọn đáp án C')

Chọn đáp án C


Câu 6: Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?

A) 55+      B) 25-34      C) 35-44      D) 45-54


In [35]:
orders_per_customer = orders.groupby('customer_id').size().reset_index(name='order_count')

cust_age = customers[customers['age_group'].notna()][['customer_id', 'age_group']]
merged   = cust_age.merge(orders_per_customer, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)

avg_orders = merged.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print('Số đơn hàng trung bình theo age_group:')
print(avg_orders.round(4).to_string())
print()
print(f'Cao nhất: {avg_orders.idxmax()} ({avg_orders.max():.4f} đơn/khách)')

Số đơn hàng trung bình theo age_group:
age_group
55+      5.4069
45-54    5.3572
35-44    5.3373
25-34    5.2452
18-24    5.2267

Cao nhất: 55+ (5.4069 đơn/khách)


In [36]:
print('Chọn đáp án A')

Chọn đáp án A


Câu 7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất?

A) West      B) Central      C) East      D) Cả ba vùng có doanh thu xấp xỉ bằng nhau


In [37]:
order_items['revenue_line'] = order_items['quantity'] * order_items['unit_price']
revenue_per_order = order_items.groupby('order_id')['revenue_line'].sum().reset_index()

orders_geo = orders[['order_id', 'zip']].merge(
    geography[['zip', 'region']], on='zip', how='left')
orders_rev = orders_geo.merge(revenue_per_order, on='order_id', how='left')

revenue_by_region = orders_rev.groupby('region')['revenue_line'].sum().sort_values(ascending=False)
print('Tổng doanh thu theo region:')
for r, v in revenue_by_region.items():
    print(f'  {r:10s}: {v:>18,.0f}')
print()
print(f'Cao nhất: {revenue_by_region.idxmax()}')

Tổng doanh thu theo region:
  East      :      7,637,532,676
  Central   :      4,941,908,472
  West      :      3,851,035,438

Cao nhất: East


In [38]:
print('Chọn đáp án A')

Chọn đáp án A


Câu 8: Trong các đơn hàng có order_status = 'Cancelled', phương thức thanh toán nào được sử dụng nhiều nhất?

A) credit_card      B) cod      C) paypal      D) bank_transfer


In [39]:
cancelled = orders[orders['order_status'].str.lower() == 'cancelled']
print(f'Tổng đơn Cancelled: {len(cancelled):,}')
print()
payment_counts = cancelled['payment_method'].value_counts()
print('Phương thức thanh toán trong đơn Cancelled:')
print(payment_counts.to_string())
print()
print(f'Phương thức thanh toán hổ biến nhất: {payment_counts.idxmax()} ({payment_counts.max():,})')


Tổng đơn Cancelled: 59,462

Phương thức thanh toán trong đơn Cancelled:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535

Phương thức thanh toán hổ biến nhất: credit_card (28,452)


In [40]:
print('Chọn đáp án A')

Chọn đáp án A


Câu 9: Trong bốn kích thước (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất (số bản ghi trong returns / số dòng trong order_items, join với products theo product_id)?

A) S      B) M      C) L      D) XL


In [41]:
sizes = ['S', 'M', 'L', 'XL']

oi_size  = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_count = oi_size[oi_size['size'].isin(sizes)].groupby('size').size()

ret_size  = returns_df.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_count = ret_size[ret_size['size'].isin(sizes)].groupby('size').size()

return_rate = (ret_count / oi_count* 100).sort_values(ascending=False)
print('Tỷ lệ trả hàng theo size (returns / order_items):')
for s, r in return_rate.items():
    print(f'  {s}: {r:.2f}% ({ret_count[s]:,} / {oi_count[s]:,})')
print()
print(f'Kích thứớc có tỉ lệ trả hàng cao nhất: {return_rate.idxmax()} ({return_rate.max():.6f})')

Tỷ lệ trả hàng theo size (returns / order_items):
  S: 5.65% (9,723 / 172,042)
  L: 5.62% (9,741 / 173,174)
  M: 5.57% (9,820 / 176,428)
  XL: 5.52% (10,655 / 193,025)

Kích thứớc có tỉ lệ trả hàng cao nhất: S (5.651527)


In [42]:
print('Chọn đáp án A')

Chọn đáp án A


Câu 10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

A) 1 kỳ (trả một lần)      B) 3 kỳ      C) 6 kỳ      D) 12 kỳ


In [43]:
avg_by_install = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print('Giá trị thanh toán trung bình theo số kỳ trả góp:')
print(avg_by_install.to_string())
print()
print(f'Kế hoạch có giá trị thanh toán cao nhất: {avg_by_install.idxmax()} kỳ ({avg_by_install.max():.2f})')

Giá trị thanh toán trung bình theo số kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729

Kế hoạch có giá trị thanh toán cao nhất: 6 kỳ (24446.65)


In [44]:
print('Chọn đáp án C')

Chọn đáp án C
